### Setting parameters

In [30]:
# ============================================================
# Extended delta-map: single-seed runner for Jupyter
# - seed を単独で渡して 1回だけ回す
# - iterative_minimization(set_params, fit_params, chol) を呼ぶ
# - 返り値は fit_params.r_est のみ
# ============================================================

using PyCall
using NPZ
@pyimport healpy as hp

include("../src/function/extended_delta_map.jl")
include("../src/function/r_estimate_extended.jl")
include("../src/function/set_data_model.jl")

# ----------------------------
# Run settings (edit here)
# ----------------------------
NSIDE = 4
LMIN  = 2
LMAX  = 2 * NSIDE

SEEDS_PER_R = 1000
R_VALUES    = [0.01]

COV_BASE = "/Users/ikumakiyoshi/Library/Mobile Documents/com~apple~CloudDocs/study_fg_rm/program/julia_Delta_map/Delta_map/make_cmb_covariance_matrix/covariance_matrix/"
COV_TAG  = "wo_beam_wo_wl"
FWHM_CON = 0.0

FREQ_BANDS  = [100, 119, 140, 166, 195, 235, 280, 337, 402]
WHICH_MODEL = "d1"
NUM_I       = 6

MASK_PATH = "../mask/P06_nside_$(NSIDE).fits"

# ----------------------------
# Helpers
# ----------------------------
function load_cov_mats(nside::Int, lmin::Int, lmax::Int; base::String=COV_BASE, tag::String=COV_TAG)
    scal_path = joinpath(base, "paper_smoothing_cov_mat_scal_nside_$(nside)_lmin_$(lmin)_lmax_$(lmax)_$(tag).npy")
    tens_path = joinpath(base, "paper_smoothing_cov_mat_tens_nside_$(nside)_lmin_$(lmin)_lmax_$(lmax)_$(tag).npy")
    return npzread(scal_path), npzread(tens_path)
end

"""
Extended版:
r ごとに「seed に依らない前計算」まで済ませて
(set_params, fit_params, cholesky_terms) を返す
"""
function prepare_context_for_r_extended(r_in::Float64;
        nside::Int=NSIDE, lmin::Int=LMIN, lmax::Int=LMAX,
        freq_bands::Vector{Int}=FREQ_BANDS,
        which_model::String=WHICH_MODEL, num_I::Int=NUM_I,
        mask_path::String=MASK_PATH)

    cov_mat_scal, cov_mat_tens = load_cov_mats(nside, lmin, lmax)
    mask = hp.read_map(mask_path)

    # Containers (Extended の元コードと同じ)
    Ninv_set = Matrix{Float64}[]
    m_set    = Vector{Float64}[]

    r_est0 = 0.5

    set_params_t = SetParams(freq_bands, which_model, r_in, 2, nside, num_I,
                             cov_mat_scal, cov_mat_tens, mask, m_set, Ninv_set)
    fit_params_t = FitParams(-3, 1.5, 20.1, r_est0)

    # seed に依らない前計算
    set_num_I!(set_params_t)
    set_truncate_N⁻¹!(set_params_t, lmin, lmax; tag=COV_TAG)

    # 元コード互換（global set_params を参照する設計を維持）
    global set_params = set_params_t
    global cholesky_terms = set_cholesky_terms!()
    global matrix_terms   = set_matrix_terms!()

    return set_params_t, fit_params_t#, cholesky_terms
end

"""
seed を1つだけ指定して iterative_minimization を1回回し、
最終的な r_est（fit_params.r_est）だけ返す（Extended版）
"""
function r_est_for_seed(r_in::Real, seed::Int;
        nside::Int=NSIDE, lmin::Int=LMIN, lmax::Int=LMAX,
        fwhm_con::Float64=FWHM_CON,
        freq_bands::Vector{Int}=FREQ_BANDS,
        which_model::String=WHICH_MODEL, num_I::Int=NUM_I,
        mask_path::String=MASK_PATH)

    set_params, fit_params = prepare_context_for_r_extended(
        Float64(r_in); nside, lmin, lmax,
        freq_bands, which_model, num_I, mask_path
    )

    set_params.seed = seed
    set_truncate_m_vec!(set_params, lmin, lmax, fwhm_con)

    iterative_minimization(set_params, fit_params)#, chol)

    return fit_params.r_est
end


r_est_for_seed

In [31]:
x = r_est_for_seed(0.01, 7)

Iteration 1: r = 0.01200669601035731, Likelihood = -1.7635615574070423e10
delta_like = 2.7635615574070423e10
delta_r = 0.4879933039896427
Iteration 2: r = 0.01162826138843931, Likelihood = -1.7635615604861515e10
delta_like = 30.791091918945312
delta_r = 0.00037843462191800015
Iteration 3: r = 0.011626402972956473, Likelihood = -1.7635615604862015e10
delta_like = 0.000499725341796875
delta_r = 1.858415482836051e-6


0.011626402972956473